In [8]:
import pandas as pd
import json
import time
from pathlib import Path
from pdf2image import convert_from_path
import mimetypes

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials


# ==========================
# CONFIGURATION (COMPANY GATEWAY)
# ==========================
base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ4NDU3OTIsImlhdCI6MTc2NDg0Mzk5NCwiYXV0aF90aW1lIjoxNzY0ODQzOTkxLCJqdGkiOiJlMGJmMjJhYS1kNTg0LTRmZWEtODM2ZC1lYTJmMmFjNjU1NWMiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjE5ODE4ZWIwLWUwZjEtNDUyOC1iYjJiLTNhNWQzNTcyZTA2OCIsImF0X2hhc2giOiJabE0yLUVhelNCQjcyUFBLLXpHbWZRIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiIxOTgxOGViMC1lMGYxLTQ1MjgtYmIyYi0zYTVkMzU3MmUwNjgiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.HrGdbF02pzH-775ERSgCDAppDQAd2UfeEygtstaLq8DzPzc3v80JFY42hQoGa_0TamoVCP8bYpQb_dJZplMQtg5dzYbiRpqdykrJKzc6J8j8lifj8LeRr2DrVoP4ka4rNnV-2Y6Dym5uQ1MEx3MxzdhSFG63hcP9QF7GG00tCnCGzKsqDBhNXP4SJEERgreew3PRnHZzrr5xxPilc6xAFBnSYxlN1WMVtzOXhEEENpHPe2vCeEF-x1uenKCq0GVSTJxDyqrV1_jqQC1GtUv1Zso_yIJVhplKGDLDJ6jEnkk1e8iQgezZ3ii47VE28i3LokK2zvtQPVZqEwnh-_oIFA"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)
print("Gemini (company gateway) initialized.")


# ==========================
# PROMPT (STRICT)
# ==========================
def get_dimension_prompt():
    return """
    You are an Engineering Dimension Extraction AI specialized in reading dimensioned mechanical
    drawings. You must ONLY extract values that are explicitly written on the drawing.

    SECTION 1 – TITLE BLOCK METADATA (HIGH PRIORITY):
    - title: The main drawing title from the title block.
    - drawing_number: The drawing / part / print number from the title block.

    SECTION 2 – PARAMETERS TO EXTRACT (STRICT REQUIREMENT):
    - length
    - inner_diameter
    - outer_diameter
    - material (MATL)
    - surface_area (S/A, square inches)
    - weight (WT, pounds)

    RULES (EXTREMELY IMPORTANT):
    - Read ONLY values clearly labeled in the drawing with dimension arrows or printed annotation.
    - DO NOT guess, estimate, infer, measure visually, or invent any value.
    - If a value is missing or not explicitly labeled, return its value=null and unit=null.
    - Surface area should be read ONLY if explicitly printed (such as S/A or Surface Area).
    - Weight should be read ONLY if explicitly printed (WT, Weight, lbs, pounds, etc.).
    - Material should be read ONLY if explicitly labeled (MATL, MATERIAL, etc.)
    - Title and drawing_number should be read ONLY from the title block or clearly labeled fields.
    - DO NOT create additional dimensions, properties or calculations.

    OUTPUT FORMAT (strict JSON, NO markdown, NO commentary):

    {
      "title": "SPACER RING",
      "drawing_number": "DRW-12345-A",

      "length_value": 100.0,
      "length_unit": "mm",
      "inner_diameter_value": 20.0,
      "inner_diameter_unit": "mm",
      "outer_diameter_value": 30.0,
      "outer_diameter_unit": "mm",
      "material": "Aluminum 6061",
      "surface_area_value": 23.6,
      "surface_area_unit": "in^2",
      "weight_value": 0.52,
      "weight_unit": "lb"
    }
    """
    

# ==========================
# LOW-LEVEL CALL (image bytes → JSON)
# ==========================
def _call_model_on_image_bytes(image_bytes: bytes, mime_type: str):
    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    response = client.models.generate_content(
        # if your gateway wants "models/gemini-2.5-pro", change here
        model="gemini-2.5-pro",
        contents=[
            get_dimension_prompt(),
            image_part,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        ),
    )

    raw = response.text or "{}"

    # Safety net in case gateway wraps in ```json
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0]
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0]

    return json.loads(raw)


# ==========================
# GEMINI CALL (per image path)
# ==========================
def analyze_image(image_path: str):
    print(f"Uploading {image_path} ...")

    # read file as bytes (no upload_file API in vertex mode)
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    try:
        data = _call_model_on_image_bytes(image_bytes, mime_type)
    except Exception as e:
        print("Parsing error:", e)
        data = {}

    # Enforce schema defaults (title + drawing_number first, then others)
    keys = [
        "title",
        "drawing_number",
        "length_value", "length_unit",
        "inner_diameter_value", "inner_diameter_unit",
        "outer_diameter_value", "outer_diameter_unit",
        "material",
        "surface_area_value", "surface_area_unit",
        "weight_value", "weight_unit",
    ]
    for k in keys:
        data.setdefault(k, None)

    return data


# ==========================
# MAIN PROCESSOR
# ==========================
def process_file(file_path: str):
    print(f"\n=== Processing file: {file_path} ===")
    ext = Path(file_path).suffix.lower()
    page_images = []

    if ext == ".pdf":
        print("   -> Converting PDF to images...")
        pages = convert_from_path(file_path, dpi=300)
        for i, p in enumerate(pages, start=1):
            img = f"temp_dim_{i}.png"
            p.save(img, "PNG")
            page_images.append(img)
    else:
        page_images = [file_path]

    rows = []
    for i, img in enumerate(page_images, start=1):
        print(f"--- Page {i} ---")
        result = analyze_image(img)
        result["Page"] = i
        rows.append(result)

    df = pd.DataFrame(rows)
    out_name = f"{Path(file_path).stem}_Extracted.xlsx"
    df.to_excel(out_name, index=False)

    print(f"\n✅ Excel generated: {out_name}")


print("\nDimension extraction tool ready (company gateway).")


Gemini (company gateway) initialized.

Dimension extraction tool ready (company gateway).


In [9]:
file_path = "Master.png"
process_file(file_path)


=== Processing file: Master.png ===
--- Page 1 ---
Uploading Master.png ...

✅ Excel generated: Master_MEASUREMENTS.xlsx
